# RAG Chatbot with MongoDB Logging (Fixed Imports)

Uses langgraph.checkpoint.mongodb.MongoDBSaver

In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.tools import tool
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph.message import BaseMessage, add_messages
from langchain.messages import HumanMessage
from langgraph.checkpoint.mongodb import MongoDBSaver
import os

load_dotenv()
os.environ["MONGO_URI"] = "mongodb://localhost:27017/"
os.environ["LANGSMITH_PROJECT"] = "RAG_USING_LANGGRAPH_and_LANGCHAIN"

In [ ]:
loader = PyPDFLoader("data/Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-TensorFlow.pdf")
docs = loader.load()
print(f"Pages: {len(docs)}")
splitter = RecursiveCharacterTextSplitter(chunk_overlap=100, chunk_size=800)
chunks = splitter.split_documents(docs)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embedding=embeddings)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={'k':5})

Pages: 1351


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1869.60it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
@tool
def rag_tool(query):
    result = retriever.invoke(query)
    return {
        'query': query,
        'context': [d.page_content for d in result],
        'metadata': [d.metadata for d in result]
    }
tools = [rag_tool]

In [ ]:
llm = ChatGroq(model="llama-3.1-8b-instant")
llm_with_tool = llm.bind_tools(tools)

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
from pymongo import MongoClient

In [ ]:
tool_node = ToolNode(tools)
def chat_node(state: ChatState):
    return {"messages": [llm_with_tool.invoke(state["messages"])]}

client=MongoClient(os.getenv["MONGO_URI"])

checkpointer = MongoDBSaver(
    client=client
    db_name="rag_chat",
    collection="conversations"
)

In [ ]:
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)
graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)
graph.add_edge("tools", "chat_node")
workflow = graph.compile(checkpointer=checkpointer)
workflow

In [ ]:
config = {"configurable": {"thread_id": "1"}}
print("Chat: 'quit' to exit")
while True:
    inp = input("You: ")
    if inp.lower() == 'quit': break
    result = workflow.invoke({"messages": [HumanMessage(content=inp)]}, config)
    print("Bot:", result['messages'][-1].content)

In [ ]:
from pymongo import MongoClient
client = MongoClient(os.getenv("MONGO_URI"))
db = client.rag_chat
coll = db.conversations
print("Logs:")
for doc in coll.find().sort("checkpoint.ts", -1).limit(2):
    print(doc['config']['thread_id'], len(doc.get('messages', [])))
client.close()